In [1]:
import os
import csv
import json
from kafka import KafkaProducer

In [2]:
DATA_DIR = '/data/'
TOPIC_NAME = 'data-topic'
BOOTSTRAP_SERVERS = ['kafka:9092']

FIELD_NAMES =[
    'id', 'customer_first_name', 'customer_last_name', 'customer_age',
    'customer_email', 'customer_country', 'customer_postal_code',
    'customer_pet_type', 'customer_pet_name', 'customer_pet_breed',
    'seller_first_name', 'seller_last_name', 'seller_email',
    'seller_country', 'seller_postal_code', 'product_name',
    'product_category', 'product_price', 'product_quantity',
    'sale_date', 'sale_customer_id', 'sale_seller_id',
    'sale_product_id', 'sale_quantity', 'sale_total_price',
    'store_name', 'store_location', 'store_city', 'store_state',
    'store_country', 'store_phone', 'store_email', 'pet_category',
    'product_weight', 'product_color', 'product_size', 'product_brand',
    'product_material', 'product_description', 'product_rating',
    'product_reviews', 'product_release_date', 'product_expiry_date',
    'supplier_name', 'supplier_contact', 'supplier_email',
    'supplier_phone', 'supplier_address', 'supplier_city',
    'supplier_country'
]

INT_FIELDS = {
    'id', 'customer_age', 'product_quantity', 'sale_customer_id',
    'sale_seller_id', 'sale_product_id', 'sale_quantity', 'product_reviews'
}
FLOAT_FIELDS = {
    'product_price', 'sale_total_price', 'product_weight', 'product_rating'
}

In [3]:
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    acks='all'
)

In [4]:
def clean_row(row, global_id):
    processed = {}
    for key, value in row.items():
        if not key: continue
        val = value.strip() if value else ""

        if not val or val.lower() == "null":
            processed[key] = 0 if key in INT_FIELDS else (0.0 if key in FLOAT_FIELDS else None)
            continue

        try:
            if key in INT_FIELDS: processed[key] = int(float(val))
            elif key in FLOAT_FIELDS: processed[key] = float(val)
            else: processed[key] = val
        except (ValueError, TypeError):
            processed[key] = 0 if key in INT_FIELDS else (0.0 if key in FLOAT_FIELDS else val)

    processed['id'] = global_id
    processed['sale_customer_id'] = global_id
    processed['sale_seller_id'] = global_id
    processed['sale_product_id'] = global_id
    processed['sale_supplier_id'] = global_id
    processed['sale_store_id'] = global_id

    return processed

In [5]:
try:
    files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])

    for file_idx, filename in enumerate(files):
        file_path = os.path.join(DATA_DIR, filename)

        with open(file_path, mode='r', encoding='utf-8') as csvfile:
            reader = csv.DictReader(csvfile, fieldnames=FIELD_NAMES, delimiter=',', quotechar='"')

            row_idx = 1
            for row in reader:
                if row['id'] == 'id':
                    continue

                global_id = file_idx * 1000 + row_idx

                payload = clean_row(row, global_id)
                producer.send(TOPIC_NAME, value=payload)

                row_idx += 1

    producer.flush()

except Exception as e:
    print(f"Error: {e}")